# Anatomy of a Great Prompt

Decompose prompts into role, task, context, constraints, format, and examples — then compare weak vs strong instructions locally and via the API.


## 1. Overview

This guide covers:

- Six prompt components: role, task, context, constraints, output format, examples
- A reusable checklist for rewriting vague instructions
- Local scoring of prompt completeness (no API)
- Side-by-side bad vs good prompts on the same task via Chat Completions


## 2. Motivation

Vague prompts produce vague outputs. In production, "make it better" is not testable. Breaking a prompt into explicit parts lets you version, diff, and evaluate changes like any other interface contract.

Strong prompts answer: **who** is speaking, **what** to do, **what data** to use, **what rules** apply, and **what shape** the answer must take.


## 3. Concepts

### 3.1 Glossary

| Part | Purpose |
|------|---------|
| **Role** | Persona or expertise ("You are a senior technical editor") |
| **Task** | Imperative action ("Rewrite the paragraph for clarity") |
| **Context** | Background facts, documents, or audience |
| **Constraints** | Must/must-not rules (length, tone, banned topics) |
| **Output format** | JSON, bullets, table, markdown sections |
| **Examples** | Input/output pairs demonstrating the pattern (optional here||

### 3.2 How it works

The model conditions on the entire prompt as token context. Clear structure reduces ambiguity: the model does not "see" your intent — only the tokens you provide. Explicit format strings increase the probability of parseable output.

### 3.3 When to use structured prompts

**Use for:** any production prompt, especially extraction, classification, and customer-facing copy.

**Trade-offs:** longer prompts cost more tokens. Start minimal; add constraints when evaluations fail.

**Skip heavy structure for:** exploratory brainstorming where format does not matter.


## 4. Architecture

```mermaid
flowchart TD
    role[Role persona] --> prompt[Composed prompt]
    task[Task instruction] --> prompt
    context[Context data] --> prompt
    constraints[Constraints] --> prompt
    format[Output format] --> prompt
    examples[Examples optional] --> prompt
    prompt --> llm[LLM]
    llm --> output[Structured completion]
```

### Prompt layers

```text
┌─────────────────────────────────────┐
│ Role + Task          (always)       │
├─────────────────────────────────────┤
│ Context              (when needed)  │
├─────────────────────────────────────┤
│ Constraints + Format (for reliability) │
├─────────────────────────────────────┤
│ Examples             (few-shot)     │
└─────────────────────────────────────┘
```


## 5. Local Python Examples


In [1]:
# Prompt checklist scorer — no API required
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable


@dataclass
class PromptParts:
    role: str = ""
    task: str = ""
    context: str = ""
    constraints: str = ""
    output_format: str = ""
    examples: str = ""

    def render(self) -> str:
        sections: list[str] = []
        if self.role:
            sections.append(f"Role: {self.role}")
        if self.task:
            sections.append(f"Task: {self.task}")
        if self.context:
            sections.append(f"Context:\n{self.context}")
        if self.constraints:
            sections.append(f"Constraints:\n{self.constraints}")
        if self.output_format:
            sections.append(f"Output format:\n{self.output_format}")
        if self.examples:
            sections.append(f"Examples:\n{self.examples}")
        return "\n\n".join(sections)

    def completeness(self) -> dict[str, bool]:
        return {
            "role": bool(self.role.strip()),
            "task": bool(self.task.strip()),
            "context": bool(self.context.strip()),
            "constraints": bool(self.constraints.strip()),
            "output_format": bool(self.output_format.strip()),
            "examples": bool(self.examples.strip()),
        }


BAD_PROMPT = "Fix this email."

GOOD_PROMPT = PromptParts(
    role="You are a professional customer-success editor.",
    task="Rewrite the email below to be polite, concise, and action-oriented.",
    context=(
        "Original email:\n"
        "Hey — your invoice is wrong again. Fix it ASAP or we cancel."
    ),
    constraints=(
        "- Max 120 words\n"
        "- No blame language\n"
        "- Include a clear next step"
    ),
    output_format="Return only the rewritten email body (no preamble).",
)

for label, parts in [("bad (string)", None), ("good (structured)", GOOD_PROMPT)]:
    print(f"=== {label} ===")
    if parts is None:
        text = BAD_PROMPT
        score = {"task": True}  # vague task only
    else:
        text = parts.render()
        score = parts.completeness()
    print(text)
    print(f"completeness: {score}")
    print()


=== bad (string) ===
Fix this email.
completeness: {'task': True}

=== good (structured) ===
Role: You are a professional customer-success editor.

Task: Rewrite the email below to be polite, concise, and action-oriented.

Context:
Original email:
Hey — your invoice is wrong again. Fix it ASAP or we cancel.

Constraints:
- Max 120 words
- No blame language
- Include a clear next step

Output format:
Return only the rewritten email body (no preamble).
completeness: {'role': True, 'task': True, 'context': True, 'constraints': True, 'output_format': True, 'examples': False}



### 5.1 Bad vs good prompt strings


In [2]:
# Compare vague vs structured prompts as plain strings
TOPIC = "on-call runbook for API latency spikes"

bad = f"Write something about {TOPIC}."

good = f'''Role: You are an SRE writing internal documentation.
Task: Draft an on-call runbook section for API latency spikes.
Context: Service stack is Python/FastAPI behind an ALB; metrics in Datadog.
Constraints:
- 5 numbered steps max
- Each step: action + owner role (on-call vs platform)
- Mention rollback before root-cause deep dive
Output format: Markdown with ## heading and numbered list only.
'''

print("BAD length:", len(bad), "chars")
print(bad)
print()
print("GOOD length:", len(good), "chars")
print(good)


BAD length: 61 chars
Write something about on-call runbook for API latency spikes.

GOOD length: 394 chars
Role: You are an SRE writing internal documentation.
Task: Draft an on-call runbook section for API latency spikes.
Context: Service stack is Python/FastAPI behind an ALB; metrics in Datadog.
Constraints:
- 5 numbered steps max
- Each step: action + owner role (on-call vs platform)
- Mention rollback before root-cause deep dive
Output format: Markdown with ## heading and numbered list only.



## 6. OpenAI SDK Examples

```python
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(root / ".env")
client = OpenAI()
```


In [3]:
# API comparison: weak vs structured prompt on the same input
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv
from openai import APIConnectionError, AuthenticationError, OpenAI, RateLimitError

def find_project_root(start: Path | None = None) -> Path:
    """Walk upward until requirements.txt is found (project root)."""
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find requirements.txt. Open the notebook from this repository "
        "or set the working directory to the project root."
    )

root = find_project_root()
load_dotenv(root / ".env")
client = OpenAI()
MODEL = "gpt-4o-mini"


def api_key_ready() -> bool:
    key = os.getenv("OPENAI_API_KEY", "")
    if not key.strip() or "your_openai_api_key" in key.lower():
        print("OPENAI_API_KEY not configured or still placeholder.")
        return False
    return True


def complete(system: str, user: str, *, temperature: float = 0.2) -> str | None:
    if not api_key_ready():
        return None
    try:
        response = client.chat.completions.create(
            model=MODEL,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        )
        return response.choices[0].message.content or ""
    except (AuthenticationError, RateLimitError, APIConnectionError) as exc:
        print(f"Request failed: {type(exc).__name__}: {exc}")
        return None


sample_email = (
    "Hey — your invoice is wrong again. Fix it ASAP or we cancel."
)

weak_user = f"Improve this email: {sample_email}"

strong_user = PromptParts(
    role="You are a professional customer-success editor.",
    task="Rewrite the email below to be polite, concise, and action-oriented.",
    context=f"Original email:\n{sample_email}",
    constraints="- Max 120 words\n- No blame language\n- Include a clear next step",
    output_format="Return only the rewritten email body (no preamble).",
).render()

print("=== WEAK PROMPT OUTPUT ===")
weak_out = complete("You are helpful.", weak_user)
if weak_out:
    print(weak_out)

print("\n=== STRONG PROMPT OUTPUT ===")
strong_out = complete("Follow the user instructions exactly.", strong_user)
if strong_out:
    print(strong_out)


=== WEAK PROMPT OUTPUT ===


Subject: Urgent: Invoice Correction Needed

Hi [Recipient's Name],

I hope this message finds you well. 

I wanted to bring to your attention that there are discrepancies in the latest invoice we received. Could you please review and correct it at your earliest convenience? 

If we cannot resolve this issue promptly, we may need to reconsider our current arrangement. 

Thank you for your immediate attention to this matter. 

Best regards,  
[Your Name]  
[Your Position]  
[Your Company]  
[Your Contact Information]  

=== STRONG PROMPT OUTPUT ===


Subject: Invoice Correction Needed

Dear [Recipient's Name],

I hope this message finds you well. I noticed some discrepancies in the recent invoice and would appreciate your assistance in correcting it at your earliest convenience. 

Could you please review the invoice and provide an updated version? If you need any specific details from my end, feel free to reach out.

Thank you for your prompt attention to this matter. I look forward to your response.

Best regards,  
[Your Name]  
[Your Position]  
[Your Contact Information]  


## 7. Implementation notes

1. **`PromptParts.render`** — Keeps sections consistent across notebooks and services.
2. **System vs user** — Put stable policy in `system`; put task + data in `user` (or split further in multi-turn apps).
3. **Output format** — Ask for "JSON only" or "no preamble" when downstream code parses the reply.
4. **Constraints before examples** — Rules apply to all examples; state them once at the top.
5. **Evaluation** — Compare weak vs strong on the same input with fixed `temperature` for fair A/B.


## 8. Best practices

- Write the **task** as an imperative verb phrase ("Extract", "Classify", "Summarize").
- Put **untrusted user content** in delimited blocks (`---`, XML tags) separate from instructions.
- Specify **audience** in role or context ("for a VP", "for on-call engineers").
- Define **done**: length limits, required fields, banned phrases.
- Version prompts in git; diff `PromptParts.render()` output, not just outcomes.
- Add **examples** only when zero-shot fails (see notebooks 05–07).


## 9. Common failure modes

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| Rambling essay | No length or format constraint | Add `Output format` and max words/items |
| Wrong tone | Role missing or vague | Set explicit persona and audience |
| Ignores provided data | Context buried or unclear | Label context; use delimiters |
| Unparseable JSON | Format not strict enough | "Valid JSON only, no markdown fences" |
| Inconsistent sections | Mixed instructions and data | Separate system policy from user payload |
| Prompt bloat | Every field filled "just in case" | Add sections only when evals fail |


## 10. Validation checklist

1. Run the checklist scorer; confirm `bad` lacks role, constraints, and format.
2. Run string comparison; confirm good prompt specifies role, task, context, constraints, format.
3. With a valid API key, run weak vs strong email rewrite; compare tone and length.
4. Verify strong output follows "no preamble" when that constraint is set.
5. Store your production prompt as structured parts, not a single unlabeled paragraph.


## 11. Summary

- Great prompts explicitly specify **role, task, context, constraints, and format**.
- Structure makes prompts testable, diffable, and easier to improve.
- Weak vs strong comparisons on the same input reveal whether clarity—not model size—is the bottleneck.


